In [62]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder

In [63]:
df = pd.read_csv('Churn_Modelling.csv')
df.head(

)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [64]:
df = df.drop(['RowNumber','CustomerId','Surname'],axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [65]:
label_encoder = LabelEncoder()
df['Gender'] = label_encoder.fit_transform(df['Gender'])




In [66]:
OneHot_Encoder= OneHotEncoder(handle_unknown='ignore',sparse_output=False)
geo_encoded = OneHot_Encoder.fit_transform(df[['Geography']]).toarray()
geo_encoded_df =pd.DataFrame(geo_encoded,columns=OneHot_Encoder.get_feature_names_out())
geo_encoded_df

AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

In [67]:
data = pd.concat([df.drop('Geography',axis=1),geo_encoded_df] , axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [68]:
X = data.drop('EstimatedSalary',axis=1)
y= data['EstimatedSalary']



In [69]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.20,random_state=42) 

In [70]:
scaler = StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)


In [71]:
with open('label_encoder.pkl','wb') as file:
    pickle.dump(label_encoder,file)

with open('OneHot_Encoder.pkl','wb') as file:
    pickle.dump(OneHot_Encoder,file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [72]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [73]:
model = Sequential([

    Dense(64,activation='relu',input_shape=(X_train_scaled.shape[1],)),  ##HL1
    Dense(32,activation='relu'),  ##HL2
    Dense(1) ##output layer 

])



c:\Users\adity\Desktop\Deep learning project ANN\myenv312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [74]:
model.compile(optimizer ='adam',loss = 'mean_absolute_error',metrics =['mae'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [75]:
log_dir= "Regressionlogs/fit/" +datetime.datetime.now().strftime("%Y%m%d - %H%M%S")
tensorflow_callbacks = TensorBoard(log_dir=log_dir,histogram_freq=1)



In [76]:
early_stopping_Callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [77]:
histroy = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_test_scaled,y_test),
    epochs =100,
    callbacks = [tensorflow_callbacks,early_stopping_Callback]

)

Epoch 1/100


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100371.7812 - mae: 100371.7812 - val_loss: 98485.4688 - val_mae: 98485.4688
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 99493.7188 - mae: 99493.7188 - val_loss: 96694.3594 - val_mae: 96694.3594
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 96393.2031 - mae: 96393.2031 - val_loss: 92129.0000 - val_mae: 92129.0000
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 90372.7422 - mae: 90372.7422 - val_loss: 84684.8906 - val_mae: 84684.8906
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 81788.9219 - mae: 81788.9219 - val_loss: 75340.0859 - val_mae: 75340.0859
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 71993.2500 - mae: 71993.2500 - val_loss: 66081.2891 - val_mae: 66081.2891
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 63074.0117 - mae: 63074.0117 - val_loss: 58659.8984 - val_mae: 58659.8984
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 5661

In [78]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [79]:
%tensorboard --logdir Regressionlogs/fit


Reusing TensorBoard on port 6007 (pid 11940), started 0:23:49 ago. (Use '!kill 11940' to kill it.)

In [80]:
test_loss ,test_mae = model.evaluate(X_test_scaled,y_test)

print(f'Test MAE : {test_mae}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 982us/step - loss: 50282.6797 - mae: 50282.6797
Test MAE : 50282.6796875


In [81]:
model.save("RegressionModel.h5")